Det ser ud til at være muligt at lave en form for heatmap hvor det kan vises hvor godt eller hvor dårligt det er at lægge solceller

In [1]:
"""
Data retrieval script for a solar-siting suitability analysis of Greater Copenhagen.

Pulls three layers:
1. Solar irradiance (PVGIS API — free, no key required)
2. Elevation / terrain for shading (Copernicus DEM GLO-30 — public AWS bucket, no key)
3. Land cover (CLMS / Urban Atlas — free but requires registration + manual download,
   so this script only shows how to load it once you've downloaded it)

Install dependencies:
    pip install requests rasterio numpy shapely --break-system-packages
"""

import requests
import numpy as np

# ---------------------------------------------------------------------------
# Bounding box for Greater Copenhagen (roughly Køge to Helsingør, out to Roskilde)
# Format: (min_lon, min_lat, max_lon, max_lat)
# ---------------------------------------------------------------------------
COPENHAGEN_BBOX = (12.30, 55.55, 12.75, 55.85)
COPENHAGEN_CENTER = (55.6761, 12.5683)  # lat, lon


# ---------------------------------------------------------------------------
# 1. PVGIS — solar irradiance for a single point
# ---------------------------------------------------------------------------
def get_pvgis_irradiance(lat: float, lon: float) -> dict:
    """
    Fetch monthly + annual average daily irradiation (kWh/m2/day) for a point,
    using PVGIS's free non-interactive API (no key needed).
    Docs: https://re.jrc.ec.europa.eu/api/
    """
    url = "https://re.jrc.ec.europa.eu/api/v5_2/MRcalc"
    params = {
        "lat": lat,
        "lon": lon,
        "horirrad": 1,       # horizontal irradiance
        "optrad": 1,         # also compute optimal angle irradiance
        "mstartyear": 2015,
        "mendyear": 2020,
        "outputformat": "json",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_pvgis_optimal_angle(lat: float, lon: float) -> dict:
    """
    Ask PVGIS to compute the optimal tilt/azimuth and expected annual yield
    for a fixed 1 kWp system at this location.
    """
    url = "https://re.jrc.ec.europa.eu/api/v5_2/PVcalc"
    params = {
        "lat": lat,
        "lon": lon,
        "peakpower": 1,     # 1 kWp reference system
        "loss": 14,         # typical system losses in %
        "optimalinclination": 1,
        "optimalangles": 1,
        "outputformat": "json",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


# ---------------------------------------------------------------------------
# 2. Copernicus DEM (GLO-30) — elevation for shading/slope analysis
#    Hosted openly on AWS, no authentication required.
#    Tiles are named by their SW corner, e.g. Copernicus_DSM_COG_10_N55_00_E012_00_DEM
# ---------------------------------------------------------------------------
def copernicus_dem_tile_url(lat: int, lon: int) -> str:
    """
    Build the S3/HTTPS URL for a 1x1 degree Copernicus DEM GLO-30 tile.
    lat/lon are the integer SW corner of the tile (e.g. 55, 12 for Copenhagen).
    """
    ns = f"N{lat:02d}" if lat >= 0 else f"S{abs(lat):02d}"
    ew = f"E{lon:03d}" if lon >= 0 else f"W{abs(lon):03d}"
    tile_name = f"Copernicus_DSM_COG_10_{ns}_00_{ew}_00_DEM"
    return (
        f"https://copernicus-dem-30m.s3.amazonaws.com/"
        f"{tile_name}/{tile_name}.tif"
    )


def download_copernicus_dem_tile(lat: int, lon: int, out_path: str):
    """Download a single Copernicus DEM tile covering the given SW corner."""
    url = copernicus_dem_tile_url(lat, lon)
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    with open(out_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    return out_path


# ---------------------------------------------------------------------------
# 3. Land cover — CLMS (Copernicus Land Monitoring Service)
#    No public no-auth REST API for bulk raster download; requires a free
#    account at https://land.copernicus.eu/. Two practical options:
#      a) Manually download "Urban Atlas 2018" for Copenhagen from the CLMS portal
#      b) Use their WMS service for visualization (works without login, no data download)
#    Below: loading a manually downloaded GeoTIFF/vector file once you have it.
# ---------------------------------------------------------------------------
def load_land_cover(path: str):
    """
    Load a downloaded CLMS Urban Atlas / CORINE land cover file.
    Vector data (.gpkg/.shp) -> use geopandas.
    Raster data (.tif) -> use rasterio.
    """
    import geopandas as gpd  # pip install geopandas --break-system-packages

    gdf = gpd.read_file(path)
    return gdf


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    lat, lon = COPENHAGEN_CENTER

    print("Fetching PVGIS irradiance for central Copenhagen...")
    irradiance = get_pvgis_irradiance(lat, lon)
    print(irradiance.get("outputs", {}).get("monthly", "no data"))

    print("\nFetching PVGIS optimal tilt/azimuth + yield...")
    pv = get_pvgis_optimal_angle(lat, lon)
    print(pv.get("outputs", {}).get("mounting_system", "no data"))

    print("\nCopernicus DEM tile URL for this area:")
    print(copernicus_dem_tile_url(55, 12))

    # Uncomment to actually download the DEM tile (~100-200MB per 1x1 degree tile):
    # download_copernicus_dem_tile(55, 12, "copenhagen_dem.tif")

Fetching PVGIS irradiance for central Copenhagen...
[{'year': 2005, 'month': 1, 'H(h)_m': 16.16, 'H(i_opt)_m': 37.91}, {'year': 2005, 'month': 2, 'H(h)_m': 24.47, 'H(i_opt)_m': 42.44}, {'year': 2005, 'month': 3, 'H(h)_m': 76.45, 'H(i_opt)_m': 108.53}, {'year': 2005, 'month': 4, 'H(h)_m': 133.61, 'H(i_opt)_m': 164.93}, {'year': 2005, 'month': 5, 'H(h)_m': 155.68, 'H(i_opt)_m': 162.79}, {'year': 2005, 'month': 6, 'H(h)_m': 172.1, 'H(i_opt)_m': 170.58}, {'year': 2005, 'month': 7, 'H(h)_m': 158.65, 'H(i_opt)_m': 157.53}, {'year': 2005, 'month': 8, 'H(h)_m': 140.03, 'H(i_opt)_m': 161.09}, {'year': 2005, 'month': 9, 'H(h)_m': 94.22, 'H(i_opt)_m': 124.65}, {'year': 2005, 'month': 10, 'H(h)_m': 61.24, 'H(i_opt)_m': 111.48}, {'year': 2005, 'month': 11, 'H(h)_m': 19.73, 'H(i_opt)_m': 41.97}, {'year': 2005, 'month': 12, 'H(h)_m': 10.57, 'H(i_opt)_m': 25.3}, {'year': 2006, 'month': 1, 'H(h)_m': 13.0, 'H(i_opt)_m': 24.76}, {'year': 2006, 'month': 2, 'H(h)_m': 27.36, 'H(i_opt)_m': 47.57}, {'year': 2

# Solar Cells + ESA (and other) Data — Overview of Possibilities

A map of the different ways satellite/Earth-observation data can be combined with
solar cell projects, beyond the siting analysis already in progress.

---

## 1. Site Suitability & Planning

**Goal:** find and rank the best locations for solar installations.

- **Land cover classification** — Sentinel-2 imagery + Copernicus Land Monitoring
  Service (CORINE / Urban Atlas) to identify brownfields, industrial rooftops,
  unused agricultural land, or land conflicting with nature protection.
- **Elevation & shading** — Copernicus DEM (GLO-30) to model terrain shadows,
  slope, and aspect for tilt/azimuth optimization.
- **Rooftop-level detail** — Sentinel-2 is too coarse (10m) for individual
  roofs; combine with national LiDAR/aerial imagery (e.g. Danish
  *Danmarks Højdemodel*) for building-level suitability.
- **Suitability heatmaps** — weighted scoring of irradiance + land use +
  shading + exclusion zones into a single map (the project already underway).

## 2. Solar Resource Assessment

**Goal:** estimate how much energy a site will actually produce.

- **Surface irradiance** — Copernicus Atmosphere Monitoring Service (CAMS)
  Radiation Service (used by PVGIS) for historical and typical-year
  solar radiation per location.
- **Aerosol & atmospheric effects** — Sentinel-5P (atmospheric composition)
  to account for local air pollution/aerosol loading that reduces irradiance,
  relevant near cities or industrial areas.
- **Cloud cover patterns** — Sentinel-3 (ocean & land colour/temperature
  instruments also carry cloud data) or EUMETSAT geostationary data for
  regional cloud climatology.

## 3. Performance Monitoring (after installation)

**Goal:** track how existing solar assets are performing.

- **Soiling & degradation detection** — very-high-res optical imagery (not
  ESA, but Planet Labs / Maxar) to spot dust, snow cover, or panel damage
  on large solar farms.
- **Thermal anomaly detection** — Sentinel-3 or Landsat thermal bands can
  flag malfunctioning panel strings (hot spots) on utility-scale farms.
- **Production forecasting / nowcasting** — combining near-real-time
  Sentinel-2/3 cloud data with weather models to predict short-term output
  for grid balancing.

## 4. Environmental & Land-Use Impact

**Goal:** check for conflicts or trade-offs.

- **Biodiversity/protected area overlap** — Copernicus Land Monitoring
  Service protected-area layers, to avoid siting on nature reserves or
  high-value farmland.
- **Land surface temperature effects** — Sentinel-3 LST data to study
  local heating/cooling effects of large solar farms over time.
- **Vegetation health nearby** — Sentinel-2 NDVI time series, useful for
  agrivoltaic (combined farming + solar) planning.

## 5. Grid & Infrastructure Context

**Goal:** factor in feasibility beyond raw solar potential.

- **Proximity to grid infrastructure** — usually from national/open data
  (e.g. OpenStreetMap power infrastructure tags), not ESA, but a key
  factor for whether a high-scoring site is actually viable.
- **Population/demand proximity** — Copernicus Global Human Settlement
  Layer, useful for distributed/rooftop solar siting near demand centers.

---

## Other (non-ESA) Data Sources Worth Combining

| Source | What it provides | Notes |
|---|---|---|
| **PVGIS** (EU/JRC) | Irradiance, optimal tilt/azimuth, yield estimates | Free API, already in use |
| **NASA POWER** | Global solar/meteorological data | Good cross-check against CAMS |
| **NREL NSRDB** | High-res solar data | US-focused, less relevant for Denmark |
| **OpenStreetMap** | Building footprints, land use, power grid lines | Free, good for local context |
| **National met offices** (DMI in Denmark) | Local climate normals, cloud cover | More precise for local calibration |
| **Danish national LiDAR/height data** | Building-level roof geometry | Needed for rooftop-scale precision |
| **Planet Labs / Maxar** (commercial) | Very high-res imagery | For rooftop or performance monitoring beyond Sentinel-2's 10m |

---

## Suggested Next Layer to Add (beyond current heatmap project)

Given the Copenhagen project is already combining irradiance + land cover +
DEM, the next-highest-value additions would likely be:

1. **OpenStreetMap grid infrastructure** — cheap to add, meaningfully improves
   feasibility ranking beyond pure physical potential.
2. **Sentinel-5P aerosol data** — refines irradiance estimates in urban areas.
3. **Danish LiDAR height data** — enables a rooftop-level version of the
   analysis, not just open land.